#### The flow 
we crate tools first then we creat the subagents in atool format then call the previous tools inside them then we crreate our main agent with its model system prompt and tools(subagents)
refine this flow en tirret format 

### Multi-Agent Tiered Architecture (2.3 pattern)

#### Tier 3 — MAIN / ORCHESTRATOR AGENT (the ROUTER)

```python
main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],   # catalogue = subagent-wrappers ONLY
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.",
)
```

- Catalogue holds the **subagent-wrappers only** — never the leaf tools.
- Routes each question to the subagent whose **name + description** match the need.

#### Tier 2 — SUBAGENT AS TOOL (the BRIDGE)

```python
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 to calculate the SQUARE ROOT of a number"""   # ← the menu
    response = subagent_1.invoke({
        "messages": [HumanMessage(content=f"Calculate the square root of {x}")]
    })
    return response["messages"][-1].content
```

- An agent is **not** a tool — this wrapper converts it into one.
- It internally runs the subagent and forwards the subagent's last message back up.

#### Tier 1 — SUBAGENTS (specialized WORKERS)

```python
subagent_1 = create_agent(model=model, tools=[square_root])
subagent_2 = create_agent(model=model, tools=[square])
```

- Each owns **one specialty** (one leaf-tool domain).
- Each runs its **own inner loop**: `model → leaf tool → model`.

#### Tier 0 — LEAF TOOLS (real abilities)

```python
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2
```

- The smallest capability units — called only by their Tier-1 subagent.

#### Data flow

```
Question: "What is the square root of 456?"
  1. TIER 3  main agent reads catalogue → desc matches "square root"
  2.        → calls call_subagent_1(456)           (main does NOT do math)
  3. TIER 1  subagent_1 runs its loop → square_root(456)              ← TIER 0
  4.        → returns result up to the wrapper
  5. TIER 2  wrapper returns response["messages"][-1].content → main
  6. TIER 3  main formats the final answer to the user
```

#### Rules of the stack

| Rule | Why |
|---|---|
| Agent ≠ Tool, wrap it | `create_agent` can't eat a subagent directly |
| Routing by description | docstrings are the menu the main agent browses |
| Each tier owns its loop | Tier 1 runs its own `model→tool→model` cycle |
| One leaf-tool = one subagent | separation of concern; main gets capability, leaf gets implementation |

Question: "What is the square root of 456?"
  1. TIER 3   main agent reads its catalogue → description matches "square root"
  2.        → calls call_subagent_1(456)                 (main does NOT do math)
  3. TIER 1   subagent_1 runs its own loop → calls square_root(456)   (Tier 0)
  4.        → returns "21.35…" up to the wrapper
  5. TIER 2   wrapper returns response["messages"][-1].content → back to main
  6. TIER 3   main formats the final answer to the user

In [16]:
from dotenv import load_dotenv

load_dotenv()

True

### Create the model

In [17]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

## Creating subagents

In [18]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [19]:
from langchain.agents import create_agent
import os
# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [20]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [21]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart i

In [22]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='25bd0858-3700-46af-9827-85e8a3729e43'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}, '__gemini_function_call_thought_signatures__': {'call_395827': 'EnEKbwFpFH0T4NiIs2+vXYHpAByBm3VEj1A/JU1hhNCt9i9J92hHvuKvNopjFp2XGrSfXGN4cxvkxMXcJoK2WSpLaLbqaM7Hoov3A9728uuBT36lH+pkyZFhsY0DzLCUXYFmSp5I6AhBnaFn9FwUaKSuyA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d365-0d47-7c73-ac1c-d0710b77cde8-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_395827', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 19, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content=[{'type': 't

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
